In [1]:
import pandas as pd
import numpy as np
import time
import unittest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Reused from Lab 05 (your own code, unchanged) ---
df = pd.read_csv(r'C:\Users\chigi\OneDrive\mlprojectdataset.csv', header=1)
df = df.drop(columns=['id'])
X = df.drop(columns=['class'])
y = df['class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)
y_train_arr = y_train.values
y_test_arr = y_test.values

In [2]:
# --- Reused from Lab 05: cal_dist, sort_distances, get_k_neighbors, vote, MyKNN ---
def cal_dist(train_data, test_point, metric='euclidean'):
    if metric == 'euclidean':
        return np.sqrt(np.sum((train_data - test_point) ** 2, axis=1))
    elif metric == 'manhattan':
        return np.sum(np.abs(train_data - test_point), axis=1)
    else:
        raise ValueError('Unknown metric: ' + metric)

def merge_sort(distances, indices=None):
    if indices is None:
        indices = np.arange(len(distances))
    if len(distances) <= 1:
        return distances, indices
    mid = len(distances) // 2
    left_d, left_i = merge_sort(distances[:mid], indices[:mid])
    right_d, right_i = merge_sort(distances[mid:], indices[mid:])
    merged_d, merged_i = [], []
    i = j = 0
    while i < len(left_d) and j < len(right_d):
        if left_d[i] <= right_d[j]:
            merged_d.append(left_d[i]); merged_i.append(left_i[i]); i += 1
        else:
            merged_d.append(right_d[j]); merged_i.append(right_i[j]); j += 1
    merged_d.extend(left_d[i:]); merged_i.extend(left_i[i:])
    merged_d.extend(right_d[j:]); merged_i.extend(right_i[j:])
    return np.array(merged_d), np.array(merged_i)

def get_k_neighbors(sorted_dist, sorted_idx, k):
    kth_distance = sorted_dist[k - 1]
    tie_mask = sorted_dist == kth_distance
    tie_positions = np.where(tie_mask)[0]
    return sorted_dist[:k], sorted_idx[:k]

def vote(neighbor_dist, neighbor_idx, y_train_arr, weighted=False):
    neighbor_labels = y_train_arr[neighbor_idx]
    classes = np.unique(neighbor_labels)
    if weighted:
        weights = 1 / (neighbor_dist + 1e-8)
        class_scores = {c: weights[neighbor_labels == c].sum() for c in classes}
    else:
        class_scores = {c: np.sum(neighbor_labels == c) for c in classes}
    max_score = max(class_scores.values())
    tied = [c for c, s in class_scores.items() if s == max_score]
    return min(tied) if len(tied) > 1 else tied[0]

class MyKNN:
    def __init__(self, n_neighbors=5, metric='euclidean', weighted=False):
        self.k = n_neighbors
        self.metric = metric
        self.weighted = weighted

    def fit(self, X_train, y_train):
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)
        return self

    def _predict_one(self, test_point):
        distances = cal_dist(self.X_train, test_point, metric=self.metric)
        sorted_dist, sorted_idx = merge_sort(distances)
        neighbor_dist, neighbor_idx = get_k_neighbors(sorted_dist, sorted_idx, self.k)
        return vote(neighbor_dist, neighbor_idx, self.y_train, weighted=self.weighted)

    def predict(self, X_test):
        X_test = np.array(X_test)
        return np.array([self._predict_one(pt) for pt in X_test])

    def score(self, X_test, y_test):
        y_test = np.array(y_test)
        preds = self.predict(X_test)
        return np.mean(preds == y_test)

In [3]:
# ===== Generated with: Claude (Anthropic) =====
# Modular kNN, independently implemented for comparison against MyKNN (Lab05, hand-written)
# and sklearn's KNeighborsClassifier.

def ai_euclidean_distance(X_train, x_test):
    """Vectorized Euclidean distance from one test point to all training points."""
    return np.linalg.norm(X_train - x_test, axis=1)

def ai_quicksort(distances, indices=None):
    """Quicksort (3rd distinct sorting algorithm, alongside bubble/insertion/merge from Lab05)."""
    if indices is None:
        indices = np.arange(len(distances))
    if len(distances) <= 1:
        return distances, indices
    pivot = distances[len(distances) // 2]
    left_mask = distances < pivot
    mid_mask = distances == pivot
    right_mask = distances > pivot

    left_d, left_i = ai_quicksort(distances[left_mask], indices[left_mask])
    right_d, right_i = ai_quicksort(distances[right_mask], indices[right_mask])

    merged_d = np.concatenate([left_d, distances[mid_mask], right_d])
    merged_i = np.concatenate([left_i, indices[mid_mask], right_i])
    return merged_d, merged_i

def ai_get_neighbors(sorted_dist, sorted_idx, k):
    """Return k nearest neighbors; ties at the k-th boundary are broken by keeping sorted order."""
    return sorted_dist[:k], sorted_idx[:k]

def ai_majority_vote(neighbor_dist, neighbor_idx, y_train_arr, weighted=False):
    """Majority (or inverse-distance-weighted) vote with lowest-label tie-breaking."""
    labels = y_train_arr[neighbor_idx]
    classes, counts = np.unique(labels, return_counts=True)
    if weighted:
        weights = 1.0 / (neighbor_dist + 1e-8)
        scores = np.array([weights[labels == c].sum() for c in classes])
    else:
        scores = counts.astype(float)
    top_score = scores.max()
    tied_classes = classes[scores == top_score]
    return tied_classes.min()

class AIKNN:
    """AI-generated kNN classifier, structurally independent from MyKNN (Lab05)."""
    def __init__(self, n_neighbors=5, weighted=False):
        self.k = n_neighbors
        self.weighted = weighted

    def fit(self, X_train, y_train):
        self.X_train = np.asarray(X_train)
        self.y_train = np.asarray(y_train)
        return self

    def _predict_single(self, x_test):
        dist = ai_euclidean_distance(self.X_train, x_test)
        sorted_dist, sorted_idx = ai_quicksort(dist)
        n_dist, n_idx = ai_get_neighbors(sorted_dist, sorted_idx, self.k)
        return ai_majority_vote(n_dist, n_idx, self.y_train, weighted=self.weighted)

    def predict(self, X_test):
        X_test = np.asarray(X_test)
        return np.array([self._predict_single(x) for x in X_test])

    def score(self, X_test, y_test):
        y_test = np.asarray(y_test)
        return np.mean(self.predict(X_test) == y_test)


# Sanity check
ai_model = AIKNN(n_neighbors=5, weighted=False)
ai_model.fit(X_train, y_train)
print("AIKNN (k=5, unweighted) accuracy:", ai_model.score(X_test, y_test))

AIKNN (k=5, unweighted) accuracy: 0.8502202643171806


In [4]:
# ===== Generated with: Claude (Anthropic) =====
# Unit tests for Lab 05 (MyKNN, cal_dist, merge_sort, vote) and Lab 06 (AIKNN) functions.

class TestLab05Functions(unittest.TestCase):
    def setUp(self):
        self.train = np.array([[0.0, 0.0], [1.0, 1.0], [3.0, 3.0]])
        self.test_pt = np.array([0.0, 0.0])
        self.y = np.array([0, 0, 1])

    def test_cal_dist_euclidean(self):
        d = cal_dist(self.train, self.test_pt, metric='euclidean')
        np.testing.assert_allclose(d, [0.0, np.sqrt(2), np.sqrt(18)])

    def test_cal_dist_manhattan(self):
        d = cal_dist(self.train, self.test_pt, metric='manhattan')
        np.testing.assert_allclose(d, [0.0, 2.0, 6.0])

    def test_cal_dist_invalid_metric(self):
        with self.assertRaises(ValueError):
            cal_dist(self.train, self.test_pt, metric='cosine')

    def test_merge_sort_order(self):
        d = np.array([5.0, 1.0, 3.0])
        sd, si = merge_sort(d)
        np.testing.assert_allclose(sd, [1.0, 3.0, 5.0])
        np.testing.assert_array_equal(si, [1, 2, 0])

    def test_get_k_neighbors(self):
        sd = np.array([0.0, 1.0, 2.0, 3.0])
        si = np.array([0, 1, 2, 3])
        nd, ni = get_k_neighbors(sd, si, k=2)
        np.testing.assert_array_equal(ni, [0, 1])

    def test_vote_majority(self):
        result = vote(np.array([1.0, 1.0, 2.0]), np.array([0, 1, 2]), self.y, weighted=False)
        self.assertEqual(result, 0)  # labels [0,0,1] -> majority 0

    def test_vote_weighted(self):
        # closer point (dist=0.1, label 1) should dominate weighted vote
        d = np.array([0.1, 5.0])
        idx = np.array([2, 0])
        y = np.array([0, 0, 1])
        result = vote(d, idx, y, weighted=True)
        self.assertEqual(result, 1)

    def test_myknn_fit_predict_shape(self):
        model = MyKNN(n_neighbors=1)
        model.fit(self.train, self.y)
        preds = model.predict(self.train)
        self.assertEqual(len(preds), len(self.train))

    def test_myknn_perfect_on_train_k1(self):
        model = MyKNN(n_neighbors=1)
        model.fit(self.train, self.y)
        preds = model.predict(self.train)
        np.testing.assert_array_equal(preds, self.y)


class TestLab06AIFunctions(unittest.TestCase):
    def setUp(self):
        self.train = np.array([[0.0, 0.0], [1.0, 1.0], [3.0, 3.0]])
        self.y = np.array([0, 0, 1])

    def test_ai_euclidean_distance(self):
        d = ai_euclidean_distance(self.train, np.array([0.0, 0.0]))
        np.testing.assert_allclose(d, [0.0, np.sqrt(2), np.sqrt(18)])

    def test_ai_quicksort_order(self):
        d = np.array([5.0, 1.0, 3.0])
        sd, si = ai_quicksort(d)
        np.testing.assert_allclose(sd, [1.0, 3.0, 5.0])
        np.testing.assert_array_equal(si, [1, 2, 0])

    def test_ai_quicksort_matches_merge_sort(self):
        d = np.random.RandomState(0).rand(50)
        sd1, si1 = ai_quicksort(d.copy())
        sd2, si2 = merge_sort(d.copy())
        np.testing.assert_allclose(sd1, sd2)
        np.testing.assert_array_equal(si1, si2)

    def test_ai_majority_vote(self):
        result = ai_majority_vote(np.array([1.0, 1.0, 2.0]), np.array([0, 1, 2]), self.y, weighted=False)
        self.assertEqual(result, 0)

    def test_aiknn_perfect_on_train_k1(self):
        model = AIKNN(n_neighbors=1)
        model.fit(self.train, self.y)
        preds = model.predict(self.train)
        np.testing.assert_array_equal(preds, self.y)

    def test_aiknn_matches_myknn_on_random_data(self):
        rng = np.random.RandomState(1)
        Xr = rng.rand(30, 3)
        yr = rng.randint(0, 2, 30)
        m1 = MyKNN(n_neighbors=3).fit(Xr, yr)
        m2 = AIKNN(n_neighbors=3).fit(Xr, yr)
        np.testing.assert_array_equal(m1.predict(Xr), m2.predict(Xr))


# Run tests in notebook (use argv trick so it doesn't try to parse Jupyter's own args)
unittest.main(argv=[''], exit=False, verbosity=2)

test_cal_dist_euclidean (__main__.TestLab05Functions.test_cal_dist_euclidean) ... ok
test_cal_dist_invalid_metric (__main__.TestLab05Functions.test_cal_dist_invalid_metric) ... ok
test_cal_dist_manhattan (__main__.TestLab05Functions.test_cal_dist_manhattan) ... ok
test_get_k_neighbors (__main__.TestLab05Functions.test_get_k_neighbors) ... ok
test_merge_sort_order (__main__.TestLab05Functions.test_merge_sort_order) ... ok
test_myknn_fit_predict_shape (__main__.TestLab05Functions.test_myknn_fit_predict_shape) ... ok
test_myknn_perfect_on_train_k1 (__main__.TestLab05Functions.test_myknn_perfect_on_train_k1) ... ok
test_vote_majority (__main__.TestLab05Functions.test_vote_majority) ... ok
test_vote_weighted (__main__.TestLab05Functions.test_vote_weighted) ... ok
test_ai_euclidean_distance (__main__.TestLab06AIFunctions.test_ai_euclidean_distance) ... ok
test_ai_majority_vote (__main__.TestLab06AIFunctions.test_ai_majority_vote) ... ok
test_ai_quicksort_matches_merge_sort (__main__.TestLab0

In [5]:
# ===== A3: Performance comparison — MyKNN (yours) vs sklearn vs AIKNN =====

def evaluate_model(model_name, fit_predict_fn, n_runs=10):
    """fit_predict_fn: callable that fits+predicts and returns y_pred. Timed over n_runs."""
    times = []
    y_pred = None
    for _ in range(n_runs):
        start = time.perf_counter()
        y_pred = fit_predict_fn()
        times.append(time.perf_counter() - start)

    acc = accuracy_score(y_test_arr, y_pred)
    prec = precision_score(y_test_arr, y_pred, average='binary', zero_division=0)
    rec = recall_score(y_test_arr, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_test_arr, y_pred, average='binary', zero_division=0)
    avg_time = np.mean(times)

    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F-score': round(f1, 4),
        'Avg Time (s, 10 runs)': round(avg_time, 5)
    }

K = 5  # fixed k for the comparison; change if your report uses a different k

def run_myknn():
    m = MyKNN(n_neighbors=K, weighted=False)
    m.fit(X_train, y_train_arr)
    return m.predict(X_test)

def run_sklearn():
    sk = KNeighborsClassifier(n_neighbors=K)
    sk.fit(X_train, y_train_arr)
    return sk.predict(X_test)

def run_aiknn():
    a = AIKNN(n_neighbors=K, weighted=False)
    a.fit(X_train, y_train_arr)
    return a.predict(X_test)

results = [
    evaluate_model('MyKNN (Lab05, hand-written)', run_myknn),
    evaluate_model('sklearn KNeighborsClassifier', run_sklearn),
    evaluate_model('AIKNN (Lab06, GenAI-generated)', run_aiknn),
]

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                         Model  Accuracy  Precision  Recall  F-score  Avg Time (s, 10 runs)
   MyKNN (Lab05, hand-written)    0.8502     0.8497  0.9704   0.9061                3.40472
  sklearn KNeighborsClassifier    0.8502     0.8497  0.9704   0.9061                0.12540
AIKNN (Lab06, GenAI-generated)    0.8502     0.8497  0.9704   0.9061                3.24698
